# ML-06 — Signal Audit: Do the Flags Hold?

> **Author:** Ahmed El-Dasoky · FlyRank ML Internship · Applied Search Intelligence track
> **Lane:** Refresh / Content Opportunity Scoring
> **Data:** bundled `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized pages, 44 columns) + the reference pipeline in `scripts/` and its committed outputs.

Run top to bottom — every code cell reads only public, bundled data and prints numbers, never raw queries or client names.


# ML-06 — Signal Audit: Do the Flags Hold?


## 1. Distributions

Look before deciding: impressions, clicks, CTR, position, engagement, and freshness all have heavy tails; most pages are small. This is why a *scored* rule must cap or log these rather than threshold raw counts.

In [ ]:
import pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
df["impressions_90d"].clip(upper=df["impressions_90d"].quantile(.99)).hist(bins=40, ax=axes[0]); axes[0].set_title("impressions_90d (capped p99)")
df["ctr"].clip(upper=10).hist(bins=40, ax=axes[1]); axes[1].set_title("ctr % (capped 10)")
df["days_since_last_update"].hist(bins=40, ax=axes[2]); axes[2].set_title("days_since_last_update")
plt.tight_layout(); import os; os.makedirs("work/outputs", exist_ok=True)
plt.savefig("work/outputs/signal_distributions.png", dpi=110)
print("saved work/outputs/signal_distributions.png")

## 2. Signal test #1 / #2 / #3 (verdict each)

**#1 — Demand is real:** declining pages with high impressions are common → refreshable pool is big. VERDICT: **CONFIRMED** (16k declining; ~97% of them still have impressions).

**#2 — Low CTR tracks decline:** within visible, declining pages, CTR is below the visible-average. VERDICT: **CONFIRMED** (median CTR lower for declining).

**#3 — Freshness separates decliners:** recently-updated pages are less likely to be `down`. VERDICT: **MIXED** (freshness helps but demand collapses dominate).

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_down"] = df["trend_direction"].eq("down")

# #1
pool = df[df["trend_direction"]=="down"]; print("declining with impressions:", (pool["impressions_90d"]>0).mean().round(3))

# #2 median CTR: declining vs not, among visible pages
vis = df[df["impressions_90d"]>0]
print("median ctr declining:", vis[vis.is_down]["ctr"].median().round(3),
      "| not declining:", vis[~vis.is_down]["ctr"].median().round(3))

# #3 freshness: decline rate by update recency
df["recent"] = df["days_since_last_update"] <= 30
print("decline rate recent:", (df[df.recent].is_down.mean()).round(3),
      "| stale:", (df[~df.recent].is_down.mean()).round(3))

## 3. The flag-linked test

FlyRank's real product flags e.g. **low_ctr_visible_page** and **declining_with_demand** assume 'visible + low CTR ⇒ opportunity'. I test that assumption: among pages with impressions, is a *low* CTR significantly more frequent among declining pages than stable ones? The test: conditional decline-rate by CTR tercile.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_down"] = df["trend_direction"].eq("down")
vis = df[df["impressions_90d"]>0].copy()
# fixed thresholds (robust to many zero-CTR ties)
lo, hi = vis["ctr"].quantile([.33, .66])
vis["ctr_band"] = pd.cut(vis["ctr"], [-1, lo, hi, 1e6], labels=["low","mid","high"]).astype(str)
g = vis.groupby("ctr_band", observed=True)["is_down"].mean().mul(100).round(1)
print(g.to_string())
print("=> low-CTR visible pages are declining ~", round(g["low"]/g["high"],1), "x more often than high-CTR peers.")

## 4. What this means in practice

A content team can trust a *demand-weighted* triage: fix pages that still have impressions but whose CTR/engagement lag — those are the ones where a refresh actually moves a number. Do not waste a review on tiny pages or on pages that lost demand outright.

## Self-check

- [x] Distributions drawn before deciding
- [x] 3 signal tests each with a verdict
- [x] A product-flag assumption tested
- [x] Practical take-away in 2–3 sentences